In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("catalog_name","")
dbutils.widgets.text("table_name","")

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
table_name = dbutils.widgets.get("table_name")
table_name = table_name.split('_')[1]

In [0]:
print(table_name)

In [0]:
df = spark.read.table(f"{catalog_name}.bronze.{table_name}")

In [0]:
spark.sql(f"Create schema if not exists {catalog_name}.silver ")

In [0]:
class Transformed:
    def __init__(self, df):
        self.df = df
    def rank(self, partition_by):
        temp_df = self.df 
        temp_df = temp_df.withColumn("year",year(col('order_date')))
        window = Window.partitionBy(partition_by).orderBy(col('year').desc())
        temp_df = temp_df.withColumn("dense_rank", dense_rank().over(window))
        temp_df = temp_df.withColumn("rank", rank().over(window))
        temp_df = temp_df.withColumn("row_number", row_number().over(window))
        return temp_df
    def extract_domain(self,email):
        df_new = self.df.withColumn("domain", split(split(col(email),"@")[1],".")[0])
        window = Window.partitionBy("domain")
        df_new = df_new.withColumn("count_domain", count("domain").over(window))
        return df_new
    def concat_name(self):
        df_new = self.df.withColumn("fullname", concat_ws(" ", col("first_name"),col("last_name")))
        df_new = df_new.drop("first_name", "last_name")
        return df_new
    
    
    

In [0]:
if table_name == "customers":
  #create the object
  transformed = Transformed(df)
  #call the method
  df = transformed.extract_domain("email")
  df = transformed.concat_name()
  df = df.dropDuplicates()
  df.write.mode("overwrite").saveAsTable(f"{catalog_name}.silver.{table_name}")
  


In [0]:
if table_name == "orders":
    transformed = Transformed(df)
    df = transformed.rank("total_amount")
    df.write.mode("overwrite").saveAsTable(f"{catalog_name}.silver.{table_name}")


In [0]:
if table_name == "regions":
    df.write.mode("overwrite").saveAsTable(f"{catalog_name}.silver.{table_name}")

In [0]:
spark.sql(f"""CREATE OR REPLACE FUNCTION {catalog_name}.silver.discount_price(p_price DOUBLE)
          RETURNS DOUBLE
          LANGUAGE SQL
          RETURN p_price * 0.9""")

In [0]:
spark.sql(f"""create or replace function {catalog_name}.silver.upper_brand(p_brand string)
          returns string
          language python
          as $$
          return p_brand.upper()
          $$
          ;""")

In [0]:
if table_name == "products":
    df = df.withColumn("discount_price", expr(f"{catalog_name}.silver.discount_price(price)"))
    df = df.withColumn("brand", expr(f"{catalog_name}.silver.upper_brand(brand)"))
    df.write.mode("overwrite").saveAsTable(f"{catalog_name}.silver.{table_name}")